<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tidb_rag_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TiDB + PyTiDBによるRAG チュートリアル

このノートブックでは、pytidbライブラリを使い、シンプルなRAGの実装を作成していきます。架空のレシピを題材として、ユーザーの質問に適したレシピを返す簡単なチャットボットを作ります。

やることは次の 4 段階です。

1. 依存パッケージをインストールする
2. TiDB Cloud Zero でクラスタを作成する
3. レシピデータとレシピテーブルを用意する
4. レシピデータを投入する
5. ベクトル検索で関連レシピを取得する
6. LLMを使って回答する



# 1. 依存パッケージのインストール

In [ ]:
!pip install -q pytidb

In [ ]:
# Pythonバージョンの確認
import sys

print(sys.version)
assert sys.version_info >= (3, 10), "Python 3.10+ が必要です"

## 2. TiDB Cloud Zero でクラスタを作成する

[TiDB Cloud Zero]()は、サインアップ不要でクラスタを作成できるTiDB Cloudです。作成したクラスタはデフォルトで30日間の有効期限があり、30日を超えてクラスタを維持する場合には、TiDB Cloud Starterへ移行することができます。
APIでクラスタの作成が完了するため、このノートブックのように使い捨てのデータベースとして使うこともできます。

下記のコードでは、クラスタを作成後、接続情報を変数 `conn` に設定しています。
また、出力で表示されるURLを使って、クラスタをTiDB Cloud Starterクラスタに移行し、データが消えないようにもできます。

In [ ]:
from pathlib import Path
import requests

# TiDB Cloud ZeroのAPIエンドポイント
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"
DATABASE_NAME = "recipe_rag"


def request_zero_instance() -> dict:
    """ TiDB Cloud Zeroのインスタンスを作成する """
    resp = requests.post(
        ZERO_API,
        json={"tag": "recipe-rag-tutorial-colab"},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance()
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Database : {DATABASE_NAME}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)

## 3. レシピデータとレシピテーブルを用意する

TiDBにレシピデータを投入します。レシピデータは `RECIPES` 変数に定義しています。

RAGではセマンティック検索（意味が近いものを検索）がよく利用されます。このとき利用されるのがベクトル型です。ベクトル型では、テキストをベクトルに変換して保存します（このことを`embedding`と呼びます）

データを保存するために、TiDBにテーブルを定義する必要があります。通常はSQLを使ってテーブルを作成しますが、pytidbライブラリを利用すると、Pythonのクラスとして作成することができます。

テーブル作成の際、ベクトル型の列は元の（ベクトル化する前の）列として定義できます。pytidbではembeddingに使う関数を指定して、その結果を自動的に保存することができます（`auto embedding`）
このノートブックの例では、AWSのtitanというembeddingモデルを利用しています。


- `description` はembeddingする前のテキストが入ります
- `description_vec` は `EmbeddingFunction(...).VectorField(...)` でembeddingされた数値列（ベクトル）が入ります


In [ ]:
# レシピデータ
RECIPES = [
    {
        "name": "霧氷豆腐の白湯がけ",
        "cuisine": "和風創作",
        "ingredients": "絹ごし豆腐 1丁, 白だし 大さじ2, 鶏ガラ白湯スープ 300ml, 三つ葉 少々, ゆず皮 少々",
        "instructions": "1. 豆腐を一晩冷凍してから自然解凍し、水気を絞る。2. 温めた鶏ガラ白湯スープに白だしを加える。3. 豆腐を器に盛り、スープを注ぎ、三つ葉とゆず皮を散らす。",
        "description": "凍らせて解凍した豆腐の独特な食感と、まろやかな白湯スープが調和する温かく軽やかな一品。寒い夜にさっぱりと食べたいときに最適な、低カロリーの創作和食。",
    },
    {
        "name": "青空カルボナーラ",
        "cuisine": "イタリアン創作",
        "ingredients": "スパゲッティ 200g, バタフライピーパウダー 小さじ1, 卵黄 2個, ペコリーノチーズ 50g, パンチェッタ 80g, 黒胡椒 適量",
        "instructions": "1. パスタをバタフライピー入りの湯で茹でる(青く染まる)。2. 卵黄とチーズ、黒胡椒を混ぜる。3. パンチェッタを炒め、茹で上がったパスタと和え、火を止めてからソースを絡める。",
        "description": "茹で湯にバタフライピーを加えることで麺が鮮やかな青色に染まる、視覚的にインパクトのあるカルボナーラ。味わいは濃厚なクリーミー系で、SNS映えする創作パスタ。",
    },
    {
        "name": "月光麻婆冷奴",
        "cuisine": "中華創作",
        "ingredients": "絹ごし豆腐 1丁, 豆鼓 大さじ1, ココナッツミルク 100ml, 花椒 小さじ1/2, ライム果汁 小さじ1, パクチー 少々",
        "instructions": "1. 豆腐を冷蔵庫で冷やす。2. 豆鼓、ココナッツミルク、花椒、ライム果汁を混ぜてソースを作る。3. 冷奴にソースをかけ、パクチーをのせる。",
        "description": "火を使わずに作れる冷たい麻婆豆腐風の一品。ココナッツミルクのまろやかさと花椒の痺れが意外な組み合わせを生む、夏向けの東南アジア風中華創作。",
    },
    {
        "name": "銀河天津飯",
        "cuisine": "中華創作",
        "ingredients": "ご飯 1膳, 卵 2個, イカ墨ペースト 小さじ1, 蟹身 50g, 銀箔 ひとつまみ, 甘酢あん 80ml",
        "instructions": "1. 卵にイカ墨ペーストを混ぜ、蟹身と一緒にふんわり焼く。2. ご飯の上にのせる。3. 甘酢あんをかけ、銀箔を散らす。",
        "description": "卵が漆黒に染まり、銀箔がきらめく宇宙のような見た目の天津飯。味自体はオーソドックスな甘酢あんの天津飯で、特別な日のサプライズに使える創作中華。",
    },
    {
        "name": "苺と山椒のリゾット",
        "cuisine": "イタリアン創作",
        "ingredients": "米 150g, 苺 6粒, 粉山椒 小さじ1/4, 生クリーム 50ml, 玉ねぎ 1/4個, 白ワイン 50ml, パルミジャーノ 30g",
        "instructions": "1. 玉ねぎを炒め、米を加えて白ワインを注ぐ。2. 出汁を少しずつ加えながら炊く。3. 仕上げに刻んだ苺、生クリーム、パルミジャーノ、粉山椒を加えて混ぜる。",
        "description": "苺の甘酸っぱさと山椒の爽やかな痺れが意外な相性を見せる、デザート寄りの食事リゾット。前菜にも食後の一品にもなる繊細な創作イタリアン。",
    },
    {
        "name": "焙じ茶ボロネーゼ",
        "cuisine": "イタリアン創作",
        "ingredients": "タリアテッレ 200g, 合い挽き肉 200g, 焙じ茶葉 大さじ1, トマト缶 200g, 玉ねぎ 1/2個, にんにく 1片, 赤ワイン 50ml",
        "instructions": "1. 焙じ茶葉を熱湯で濃く抽出する。2. 玉ねぎとにんにくを炒め、挽き肉を加えて焼き付ける。3. 焙じ茶、トマト缶、赤ワインを加えて30分煮込み、茹でたパスタと和える。",
        "description": "焙じ茶の香ばしさとトマトの酸味が溶け合う、和洋折衷の深みのあるボロネーゼ。コーヒーや紅茶を加える伝統的レシピを焙じ茶に置き換えた創作パスタ。",
    },
    {
        "name": "雪見みたらしクレープ",
        "cuisine": "和洋折衷スイーツ",
        "ingredients": "クレープ生地 2枚, バニラアイス 100g, 白玉団子 6個, みたらしのタレ 大さじ3, きな粉 適量",
        "instructions": "1. クレープ生地を温める。2. バニラアイスと白玉団子をのせる。3. みたらしのタレをかけ、きな粉を振って包む。",
        "description": "もちもちの白玉と冷たいアイス、甘じょっぱいみたらしを薄いクレープで包んだ和洋折衷デザート。冷温のコントラストと食感の対比が楽しめる創作スイーツ。",
    },
    {
        "name": "海風ガスパチョ",
        "cuisine": "スペイン創作",
        "ingredients": "トマト 3個, きゅうり 1本, 昆布だし 200ml, 牡蠣のオイル漬け 4個, オリーブオイル 大さじ2, レモン果汁 小さじ2",
        "instructions": "1. トマト、きゅうりをざく切りにする。2. 昆布だし、牡蠣、オリーブオイル、レモン果汁と一緒にミキサーにかける。3. 冷蔵庫で2時間冷やしてから器に注ぐ。",
        "description": "スペインの冷製スープに日本の昆布だしと牡蠣の旨味を加えた、海の香り漂う創作ガスパチョ。前菜として軽く始めたい夏の食卓に向く、低カロリーな冷製料理。",
    },
    {
        "name": "百花蜂蜜キムチチゲ",
        "cuisine": "韓国創作",
        "ingredients": "白菜キムチ 200g, 豚バラ肉 150g, 百花蜂蜜 大さじ2, 木綿豆腐 1/2丁, 長ねぎ 1本, ごま油 大さじ1, だし汁 500ml",
        "instructions": "1. ごま油で豚バラを炒め、キムチを加えて軽く炒める。2. だし汁と蜂蜜を加えて煮立てる。3. 豆腐とねぎを加えて10分煮込む。",
        "description": "辛さの中に蜂蜜のまろやかな甘さが広がる、子どもでも食べやすい優しいキムチチゲ。寒い季節に体を温めたい時に作りたい、家庭向けの創作韓国料理。",
    },
    {
        "name": "森のきのこ大福",
        "cuisine": "和創作スイーツ",
        "ingredients": "白玉粉 100g, 砂糖 大さじ2, 水 120ml, あんこ 80g, 乾燥ポルチーニ茸 10g, バルサミコ酢 小さじ1",
        "instructions": "1. ポルチーニを戻して細かく刻み、バルサミコ酢と混ぜてあんこに加える。2. 白玉粉、砂糖、水を混ぜてレンジで加熱し、求肥を作る。3. 求肥でポルチーニ入りあんを包む。",
        "description": "森の香り高いポルチーニ茸とバルサミコ酢を忍ばせた、大人向けの創作大福。意外性のあるお茶請けや、ワインに合わせる和スイーツとして楽しめる。",
    },
]


In [ ]:
from pytidb import TiDBClient
from pytidb.datatype import TEXT
from pytidb.embeddings import EmbeddingFunction
from pytidb.schema import Field, TableModel

def connect() -> TiDBClient:
    """ TiDBへ接続し、コネクションを返す関数 """
    return TiDBClient.connect(
        host=conn["host"],
        port=4000,
        username=conn["username"],
        password=conn["password"],
        database=DATABASE_NAME,
        ensure_db=True,
    )

# embedding関数。ここでは AWSのTitanを利用
# TiDB Cloudの標準のものを利用するため、API_KEYなどは不要
_embed_func = EmbeddingFunction(
    model_name="tidbcloud_free/amazon/titan-embed-text-v2",
)

class Recipe(TableModel):
    """ pydanticによるTiDBのテーブル定義 """
    __tablename__ = "recipes"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    name: str = Field()
    cuisine: str = Field()
    ingredients: str = Field(sa_type=TEXT)
    instructions: str = Field(sa_type=TEXT)
    description: str = Field(sa_type=TEXT)
    # ベクトル型の列。元となるテキスト列にembedding関数を適用したものになる
    description_vec: list[float] = _embed_func.VectorField(source_field="description")


def open_recipe_table(db: TiDBClient):
    """ テーブルを作成 (CREATE TABLE ... IF NOT EXIST) し、テーブルを返す関数 """
    return db.create_table(schema=Recipe, if_exists="skip")


print(f"レシピ件数: {len(RECIPES)}")

## 4. レシピデータを投入する

レシピデータをテーブルに投入します。pytidbの `table.bulk_insert()` 関数を利用します。


In [ ]:
db = connect()
table = open_recipe_table(db)

existing = table.rows()
if existing > 0:
    print(f"既に {existing} 件のデータがあります。スキップします。")
else:
    print(f"=== {len(RECIPES)} 件のレシピを投入 ===")
    table.bulk_insert([Recipe(**r) for r in RECIPES])
    print(f"投入完了。現在のレコード数: {table.rows()}")

## 5. ベクトル検索で関連レシピを取得する

まず、TiDBだけを利用して、ベクトル検索を試します。ベクトル検索は、pytidbの `table.search(QUERY)` 関数を利用して簡単に実行できます。

ベクトル検索は、ベクトル同士の演算で質問とテーブル中のデータの「近さ」を測ります。通常、質問文をベクトル化するためにembeddingで利用したのと同じ関数でベクトル化します。

`table.search(QUERY)` 関数は、この処理を自動でやってくれるため、ユーザーがそのコードを自分で書く必要はありません。

下記のコードを実行すると、質問文と類似している（と判断された）上位3件のレコードが出力されます。


In [ ]:
QUESTION = "さっぱり食べられる和風の温かい料理は?"
RETRIEVE_LIMIT = 3

retrieved = table.search(QUESTION).limit(RETRIEVE_LIMIT).to_pydantic()

print("質問:", QUESTION)
print()
print("=== ベクトル検索ヒット ===")
for i, r in enumerate(retrieved, start=1):
    print(f"{i}. {r.name} ({r.cuisine})")
    print(f"   説明: {r.description}")
    print()

## 6. LLMを使って回答を作成する

ベクトル検索で上位N件を取得すると、中には回答としてふさわしくないものや、そのままでは親切な回答にはならないものもあります。

そこで、通常はベクトル検索の結果を含める形でプロンプトを作成して、LLMにその中から必要な情報を選択させ、適切な回答を生成させます。

まずは、ベクトル検索の結果から、LLMに与えるプロンプトを作りましょう。プロンプトには `context` としてベクトル検索の結果を埋め込み、`question`として質問文を埋め込みます。このプロンプトを元にLLMが適切な回答を生成します。



In [ ]:
PROMPT_TEMPLATE = """あなたは料理アシスタントです。以下のレシピ情報のみを根拠に、ユーザーの質問に日本語で答えてください。
回答は料理名と、説明や作り方のサマリーをつけてください。情報が不足している場合は「レシピデータに該当する情報がありません」と答えてください。

---
{context}
---

質問: {question}
"""


def build_context(recipes) -> str:
    """ ベクトル検索の結果から、プロンプトに入れるコンテキストを作成する """
    blocks = []
    for r in recipes:
        blocks.append(
            f"【{r.name}】({r.cuisine})\n"
            f"  説明: {r.description}\n"
            f"  材料: {r.ingredients}\n"
            f"  作り方: {r.instructions}"
        )
    return "\n\n".join(blocks)

def build_prompt(question,receipes) -> str:
    """ 質問文とベクトル検索の結果から、プロンプトを作成する """
    context = build_context(receipes)
    return PROMPT_TEMPLATE.format(question=question, context=context)

prompt = build_prompt(QUESTION, retrieved)
print("=== LLMプロンプト ===")

print(prompt[:2500])

このプロンプトを利用して、LLMに回答を作成させます。

このノートブックではColabで利用できるGeminiモデルを利用していますが、他のモデルでも動作します。ただし、APIの呼び出し方はモデルによって異なることがあります。


In [ ]:
from google.colab import ai

answer = ai.generate_text(prompt)
print("=== LLM 回答 ===")
print(answer.strip())

## 7. 追加実験

質問を変えて、検索上位がどう変わるかを見てください。

試しやすい例:
- `見た目がインパクトのある料理を教えて`
- `辛いけど子どもでも食べられる料理は?`
- `デザートで意外な食材を使ったものは?`
- `火を使わずに作れる料理ある?`


In [ ]:
QUESTION = "火を使わずに作れる料理ある?"
retrieved = table.search(QUESTION).limit(3).to_pydantic()
prompt = build_prompt(QUESTION, retrieved)

answer = ai.generate_text(prompt)
print("=== LLM 回答 ===")
print(answer.strip())